In [1]:
import ccxt
import talib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from backtesting.lib import crossover,cross
from backtesting import Backtest, Strategy
pd.set_option('display.max_columns', None)

%matplotlib inline

/opt/conda/miniconda3/envs/analysis/lib/python3.12/site-packages/backtesting/_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

In [5]:
def fetch_ohlcv(symbol='BTC/USDT', timeframe='15m', limit=1500, exchange_id='binance'):
    exchange_class = getattr(ccxt, exchange_id)
    exchange = exchange_class({'enableRateLimit': True})
    raw = exchange.fetch_ohlcv(symbol, timeframe=timeframe, limit=limit)
    
    # backtesting.py تتطلب أسماء أعمدة معينة وحروف كبيرة (Capitalized)
    df = pd.DataFrame(raw, columns=['Date', 'Open', 'High', 'Low', 'Close', 'Volume'])
    df['Date'] = pd.to_datetime(df['Date'], unit='ms')
    df.set_index('Date', inplace=True)
    return df

# إطار زمني 15 دقيقة (ضمن النطاق 5م - 30م+)
SYMBOL = 'ADA/USDT'
TIMEFRAME = '15m'
data = fetch_ohlcv(symbol=SYMBOL, timeframe=TIMEFRAME, limit=500, exchange_id='bybit')
print(f'تم جلب {len(data)} شمعة لـ {SYMBOL} على إطار {TIMEFRAME}')
data.tail()


تم جلب 500 شمعة لـ ADA/USDT على إطار 15m


,Open,High,Low,Close,Volume
Date,,,,,
2026-07-30 13:15:00,0.1660,0.1663,0.1656,0.1660,45673.86
2026-07-30 13:30:00,0.1660,0.1679,0.1659,0.1675,488200.25
2026-07-30 13:45:00,0.1675,0.1691,0.1675,0.1682,1367520.96
2026-07-30 14:00:00,0.1682,0.1690,0.1681,0.1687,421003.09
2026-07-30 14:15:00,0.1687,0.1689,0.1687,0.1688,61281.73


In [11]:
# ==========================================
# 1. Hull MA Helper Functions (Matching Pine Script)
# ==========================================

def compute_hma(series, length):
    half_length = int(length / 2)
    sqrt_length = int(np.round(np.sqrt(length)))
    wma_half = talib.WMA(series, half_length)
    wma_full = talib.WMA(series, length)
    return talib.WMA(2 * wma_half - wma_full, sqrt_length)

def compute_ehma(series, length):
    half_length = int(length / 2)
    sqrt_length = int(np.round(np.sqrt(length)))
    ema_half = talib.EMA(series, half_length)
    ema_full = talib.EMA(series, length)
    return talib.EMA(2 * ema_half - ema_full, sqrt_length)

def compute_thma(series, length):
    third_length = int(length / 3)
    half_length = int(length / 2)
    wma_third = talib.WMA(series, third_length)
    wma_half = talib.WMA(series, half_length)
    wma_full = talib.WMA(series, length)
    return talib.WMA(3 * wma_third - wma_half - wma_full, length)

# ==========================================
# 2. Unified Indicator Calculation Function
# ==========================================

def compute_all_indicators(df, bb_period=14, bb_dev=0.6, atr_period=1, atr_mult=1.0,
                           mom_period=10, hull_mode='Hma', hull_length=55, hull_mult=1.0):
    """
    دالة موحدة لحساب مؤشرات FLI, MOM, و Hull MA وإضافتها للبيانات
    """
    close = df['Close'].to_numpy(dtype=float)
    high = df['High'].to_numpy(dtype=float)
    low = df['Low'].to_numpy(dtype=float)

    # 1. حساب Bollinger Bands و ATR لـ FLI
    upper, mid, lower = talib.BBANDS(close, timeperiod=bb_period, nbdevup=bb_dev, nbdevdn=bb_dev)
    atr = talib.ATR(high, low, close, timeperiod=atr_period)

    # 2. حساب مؤشر MOM
    df['MOM'] = talib.MOM(close, timeperiod=mom_period)

    # 3. حساب خط المتابعة FollowLine (FLI)
    fl = np.full_like(close, np.nan)
    trend = np.zeros(len(close), dtype=int)

    curr_trend = 0
    curr_fl = np.nan

    for i in range(len(close)):
        if np.isnan(upper[i]) or np.isnan(atr[i]):
            continue

        if curr_trend == 0:
            if close[i] > upper[i]:
                curr_trend = 1
            elif close[i] < lower[i]:
                curr_trend = -1

        if curr_trend == 1:
            level = low[i] - (atr[i] * atr_mult)
            curr_fl = max(curr_fl, level) if not np.isnan(curr_fl) else level
            if close[i] < curr_fl:
                curr_trend = 0
                curr_fl = np.nan

        elif curr_trend == -1:
            level = high[i] + (atr[i] * atr_mult)
            curr_fl = min(curr_fl, level) if not np.isnan(curr_fl) else level
            if close[i] > curr_fl:
                curr_trend = 0
                curr_fl = np.nan

        trend[i] = curr_trend
        fl[i] = curr_fl

    df['FollowLine'] = fl
    df['FollowTrend'] = trend

    # 4. حساب مؤشر Hull MA
    adj_length = int(hull_length * hull_mult)
    if hull_mode == 'Hma':
        df['HullMA'] = compute_hma(close, adj_length)
    elif hull_mode == 'Ehma':
        df['HullMA'] = compute_ehma(close, adj_length)
    elif hull_mode == 'Thma':
        df['HullMA'] = compute_thma(close, adj_length)
    else:
        df['HullMA'] = np.nan

    df['HullMA_Shifted'] = df['HullMA'].shift(2)

    return df

# ==========================================
# 3. Updated Strategy with Signal Prioritization & Confirmation
# ==========================================

class CombinedMomFliHullStrategy(Strategy):
    # ==========================================
    # PUBLIC PARAMETERS - Modify these as needed
    # ==========================================
    
    # MOM Parameters
    up_zone = 0.00639
    middle_zone = 0.00001
    down_zone = -0.006660
    
    # Hull MA Parameters
    hull_mode = 'Ehma'  # Options: 'Hma', 'Ehma', 'Thma'
    hull_length = 59
    hull_mult = 0.6
    
    # Signal Priority Order (1 = highest priority)
    # Lower number = higher priority
    priority_mom = 1
    priority_fli = 2
    priority_hull = 3
    
    # Confirmation Settings
    # Timeframe threshold in minutes - if timeframe < this, wait for confirmation
    confirmation_threshold_minutes = 30
    
    # ==========================================
    
    def init(self):
        # Register indicators
        self.mom = self.I(lambda x: x, self.data.MOM, name='MOM')
        self.fli_trend = self.I(lambda x: x, self.data.FollowTrend, name='FollowTrend')
        self.fli_line = self.I(lambda x: x, self.data.FollowLine, name='FollowLine')
        self.hull_ma = self.I(lambda x: x, self.data.HullMA, name='HullMA')
        self.hull_ma_shifted = self.I(lambda x: x, self.data.HullMA_Shifted, name='HullMA_Shifted')
        
        # Signal tracking
        self.pending_signal = None
        self.pending_signal_time = None
        self.pending_signal_priority = None
        
        # Determine timeframe from data
        if hasattr(self.data, 'df') and len(self.data.df) > 1:
            time_diff = self.data.df.index[1] - self.data.df.index[0]
            self.timeframe_minutes = time_diff.total_seconds() / 60
        else:
            self.timeframe_minutes = 15
        
        # Determine if confirmation is needed
        # Only apply confirmation for timeframes < 30 minutes
        self.need_confirmation = self.timeframe_minutes < self.confirmation_threshold_minutes

    def _get_signal_priority(self, signal_type, indicator):
        """Return priority value for a signal (lower = higher priority)"""
        if indicator == 'MOM':
            return self.priority_mom
        elif indicator == 'FLI':
            return self.priority_fli
        elif indicator == 'HULL':
            return self.priority_hull
        return 999

    def next(self):
        if len(self.fli_trend) < 3:
            return

        mom_now = self.mom[-1]
        fli_trend_now = self.fli_trend[-1]
        fli_trend_prev = self.fli_trend[-2]

        hull_now = self.hull_ma[-1]
        hull_shifted = self.hull_ma_shifted[-1]
        hull_prev = self.hull_ma[-2]
        hull_shifted_prev = self.hull_ma_shifted[-2]

        # Determine Hull MA trend
        hull_trend = 1 if hull_now > hull_shifted else -1
        hull_trend_prev = 1 if hull_prev > hull_shifted_prev else -1

        # Extract MOM signals
        mom_buy = mom_now > self.up_zone
        mom_sell = mom_now < self.down_zone
        mom_exit_long = mom_now < self.middle_zone
        mom_exit_short = mom_now > self.middle_zone

        # Extract FLI signals
        fli_buy = (fli_trend_now == 1 and fli_trend_prev != 1)
        fli_sell = (fli_trend_now == -1 and fli_trend_prev != -1)
        fli_exit_long = (fli_trend_now != 1 and fli_trend_prev == 1)
        fli_exit_short = (fli_trend_now != -1 and fli_trend_prev == -1)

        # Extract Hull MA signals
        hull_buy = (hull_trend == 1 and hull_trend_prev != 1)
        hull_sell = (hull_trend == -1 and hull_trend_prev != -1)
        hull_exit_long = (hull_trend == -1 and hull_trend_prev == 1)
        hull_exit_short = (hull_trend == 1 and hull_trend_prev == -1)

        # ==========================================
        # SIGNAL PRIORITIZATION LOGIC
        # ==========================================
        
        # Collect all current signals with their priorities
        current_signals = []
        
        # MOM signals (highest priority)
        if mom_buy and hull_trend == 1:
            current_signals.append(('buy', self.priority_mom, 'MOM'))
        elif mom_sell and hull_trend == -1:
            current_signals.append(('sell', self.priority_mom, 'MOM'))
        
        # FLI signals
        if fli_buy and not mom_sell and hull_trend == 1:
            current_signals.append(('buy', self.priority_fli, 'FLI'))
        elif fli_sell and not mom_buy and hull_trend == -1:
            current_signals.append(('sell', self.priority_fli, 'FLI'))
        
        # Hull signals
        if hull_buy:
            current_signals.append(('buy', self.priority_hull, 'HULL'))
        elif hull_sell:
            current_signals.append(('sell', self.priority_hull, 'HULL'))

        # Sort by priority (lower number = higher priority)
        current_signals.sort(key=lambda x: x[1])
        
        # Get the highest priority signal
        current_signal = current_signals[0] if current_signals else None

        # ==========================================
        # CONFIRMATION LOGIC
        # ==========================================
        
        if self.need_confirmation:
            # Timeframe < 30 minutes: wait for next candle confirmation
            
            # Check if we have a pending signal from previous bar
            if self.pending_signal is not None:
                # Confirm and execute the pending signal
                if self.pending_signal == 'buy':
                    if self.position.is_short:
                        self.position.close()
                    if not self.position.is_long:
                        self.buy()
                elif self.pending_signal == 'sell':
                    if self.position.is_long:
                        self.position.close()
                    if not self.position.is_short:
                        self.sell()
                
                # Reset pending signal
                self.pending_signal = None
                self.pending_signal_time = None
                self.pending_signal_priority = None
            
            # If we have a new signal, store it for next bar confirmation
            if current_signal is not None and self.pending_signal is None:
                self.pending_signal = current_signal[0]
                self.pending_signal_time = len(self.data.Close)
                self.pending_signal_priority = current_signal[1]
        else:
            # Timeframe >= 30 minutes: execute immediately
            if current_signal is not None:
                if current_signal[0] == 'buy':
                    if self.position.is_short:
                        self.position.close()
                    if not self.position.is_long:
                        self.buy()
                elif current_signal[0] == 'sell':
                    if self.position.is_long:
                        self.position.close()
                    if not self.position.is_short:
                        self.sell()

        # ==========================================
        # EXIT LOGIC (Always check exits)
        # ==========================================
        
        if self.position.is_long:
            if mom_exit_long or fli_exit_long or hull_exit_long:
                self.position.close()
                self.pending_signal = None  # Cancel any pending signal
        elif self.position.is_short:
            if mom_exit_short or fli_exit_short or hull_exit_short:
                self.position.close()
                self.pending_signal = None  # Cancel any pending signal



In [13]:
# Calculate indicators
data = compute_all_indicators(data, hull_mode='Hma', hull_length=59, hull_mult=1.0)

# Run backtest
bt = Backtest(data, CombinedMomFliHullStrategy, cash=100, commission=0.0001, exclusive_orders=True)

stats = bt.run()
print(stats)
bt.plot()

Backtest.run:   0%|          | 0/432 [00:00<?, ?bar/s]

Start                     2026-07-25 09:30:00
End                       2026-07-30 14:15:00
Duration                      5 days 04:45:00
Exposure Time [%]                        46.0
Equity Final [$]                     92.53106
Equity Peak [$]                      100.2316
Commissions [$]                       0.66997
Return [%]                           -7.46894
Buy & Hold Return [%]                 2.17918
Return (Ann.) [%]                   -99.11048
Volatility (Ann.) [%]                 0.39487
CAGR [%]                            -99.57076
Sharpe Ratio                       -250.99728
Sortino Ratio                        -2.16942
Calmar Ratio                         -9.89905
Alpha [%]                            -7.74931
Beta                                  0.12866
Max. Drawdown [%]                   -10.01212
Avg. Drawdown [%]                    -5.10166
Max. Drawdown Duration        4 days 09:15:00
Avg. Drawdown Duration        2 days 05:08:00
# Trades                          

GridPlot(id='p4393', ...)